# **Подготовка среды выполнения**

In [ ]:
!pip install transformers datasets torch evaluate accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00


In [ ]:
import torch # тензоры, GPU, softmax, режимы train/eval
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer # модели, токенизаторы, Trainer
from datasets import load_dataset, DatasetDict, concatenate_datasets # удобное хранение/разбиение/преобразование данных
import evaluate # метрики (accuracy/F1 и т.п.)
import accelerate # упрощает распределённое обучение в библиотеке PyTorch

import collections
import numpy as np


In [ ]:
print("GPU доступен:", torch.cuda.is_available())
print("Название GPU:", torch.cuda.get_device_name(0)) # Серверная видеокарта

GPU доступен: True
Название GPU: Tesla T4


# Загрузка датасета

In [ ]:
dataset = load_dataset("imdb") # загрузка датасета

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [ ]:
dataset["train"][0] # просмотр 1 отзыва для проверки

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [ ]:
lables = dataset["train"]["label"]
counter_lables = collections.Counter(lables) # подсчет баланса таргетного поля
counter_lables

Counter({0: 12500, 1: 12500})

# Baseline

In [ ]:
sentiment_pipeline = pipeline("sentiment-analysis") # загружаем модель DistilBERT, обученную на SST-2

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [ ]:
sentiment_pipeline("This movie was absolutely amazing!")

[{'label': 'POSITIVE', 'score': 0.9998781681060791}]

In [ ]:
sentiment_pipeline(dataset["train"][0]["text"][:1642])

[{'label': 'POSITIVE', 'score': 0.7872862815856934}]

In [ ]:
for i in range(50):
  text = dataset["train"][i]["text"]
  result = sentiment_pipeline(text[:1642])
  print(f"Preview: {text[:100]}...")
  print(f"Result: {result}")



You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Preview: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it w...
Result: [{'label': 'POSITIVE', 'score': 0.7872862815856934}]
Preview: "I Am Curious: Yellow" is a risible and pretentious steaming pile. It doesn't matter what one's poli...
Result: [{'label': 'NEGATIVE', 'score': 0.9991909861564636}]
Preview: If only to avoid making this type of film in the future. This film is interesting as an experiment b...
Result: [{'label': 'NEGATIVE', 'score': 0.998217761516571}]
Preview: This film was probably inspired by Godard's Masculin, féminin and I urge you to see that film instea...
Result: [{'label': 'POSITIVE', 'score': 0.8144614100456238}]
Preview: Oh, brother...after hearing about this ridiculous film for umpteen years all I can think of is that ...
Result: [{'label': 'NEGATIVE', 'score': 0.9992231130599976}]
Preview: I would put this at the top of my list of films in the category of unwatchable trash! There are film...
Result: [{'label':

In [ ]:
for i in range(12501):
    if dataset["train"][i]["label"] == 1:
        print(dataset["train"][i]["text"][:200])
        print(sentiment_pipeline(dataset["train"][i]["text"][:512]))
        break

Zentropa has much in common with The Third Man, another noir-like film set among the rubble of postwar Europe. Like TTM, there is much inventive camera work. There is an innocent American who gets emo
[{'label': 'POSITIVE', 'score': 0.9754906296730042}]


# Подготовка данных и токенизация

## Подготовка данных

In [ ]:
seed = 42 # воспроизводимость результата

train_per_class = 2500
test_per_class  = 1000

train_ds = dataset["train"]
test_ds = dataset["test"]

# Фильтрование по классам (т.к. в датасете lables разделены на блоки)

train_neg = train_ds.filter(lambda x: x["label"] == 0)
train_pos = train_ds.filter(lambda x: x["label"] == 1)

test_neg = test_ds.filter(lambda x: x["label"] == 0)
test_pos = test_ds.filter(lambda x: x["label"] == 1)

# Перемешивание для наилучшего обучения (сбалансированный subsample)

train_neg_small = train_neg.shuffle(seed=seed).select(range(train_per_class))
train_pos_small = train_pos.shuffle(seed=seed).select(range(train_per_class))

test_neg_small = test_neg.shuffle(seed=seed).select(range(test_per_class))
test_pos_small = test_pos.shuffle(seed=seed).select(range(test_per_class))

small_train = concatenate_datasets([train_neg_small, train_pos_small]).shuffle(seed=seed)
small_test = concatenate_datasets([test_neg_small, test_pos_small]).shuffle(seed=seed)

small_dataset = DatasetDict({"train": small_train, "test": small_test})
print(small_dataset)

Filter:   0%|          | 0/25000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/25000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/25000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/25000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})


## Токенизация

In [ ]:
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint) # автоматически загружает правильный токенизатор, совместимый именно с этой моделью

# distilbert - облегчённая версия BERT
# base - базовый размер (не huge, не tiny)
# uncased - без учёта регистра (все слова приводятся к lower case)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
def tokenize_function(batch):
    print(len(batch["text"]))
    return tokenizer(
        batch["text"], # пачка примеров
        truncation=True, # обрезать текст, если он длиннее max_length
        padding="max_length", # дополнять короткие тексты до max_length нулями (спец токеном)
        max_length=256 # ограничение по длине
    )

In [ ]:
tokenized_dataset = small_dataset.map(tokenize_function, batched=True)
print(tokenized_dataset)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

1000
1000
1000
1000
1000


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

1000
1000
DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
})


In [ ]:
tokenized_dataset["train"]["input_ids"][:2]

[[101,
  1996,
  2034,
  2382,
  2781,
  1997,
  9543,
  11246,
  4665,
  2018,
  2026,
  4344,
  17170,
  17989,
  2006,
  1996,
  6556,
  1010,
  22303,
  2000,
  17312,
  2105,
  2000,
  3422,
  2242,
  2842,
  1012,
  1996,
  18458,
  1997,
  2048,
  4898,
  1010,
  2091,
  2006,
  2037,
  6735,
  1010,
  2542,
  1999,
  1037,
  2969,
  1011,
  5527,
  1011,
  2686,
  1000,
  8026,
  1000,
  2001,
  19499,
  19142,
  1010,
  2021,
  1010,
  16267,
  20857,
  1012,
  1026,
  7987,
  1013,
  1028,
  1026,
  7987,
  1013,
  1028,
  1996,
  4955,
  1997,
  1996,
  2839,
  1010,
  2209,
  2011,
  3533,
  6090,
  3406,
  15204,
  2080,
  1011,
  1996,
  2502,
  3066,
  3185,
  3124,
  1010,
  2008,
  3268,
  1999,
  1996,
  2380,
  1998,
  25126,
  1999,
  1037,
  13697,
  7062,
  1010,
  3253,
  3246,
  1998,
  1045,
  2787,
  2000,
  2507,
  2009,
  1037,
  2261,
  2062,
  2781,
  1012,
  1998,
  2059,
  1037,
  2261,
  2062,
  2127,
  19031,
  3723,
  26699,
  5644,
  4955,
  2004,
  

## Подготовка датасета под Trainer

In [ ]:
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset = tokenized_dataset.rename_column("label", "labels") # Transformers ожидает, что таргет будет называться "labels"

In [ ]:
tokenized_dataset.set_format("torch") # перевод в формат PyTorch

In [ ]:
tokenized_dataset["train"][0]

{'labels': tensor(0),
 'input_ids': tensor([  101,  1996,  2034,  2382,  2781,  1997,  9543, 11246,  4665,  2018,
          2026,  4344, 17170, 17989,  2006,  1996,  6556,  1010, 22303,  2000,
         17312,  2105,  2000,  3422,  2242,  2842,  1012,  1996, 18458,  1997,
          2048,  4898,  1010,  2091,  2006,  2037,  6735,  1010,  2542,  1999,
          1037,  2969,  1011,  5527,  1011,  2686,  1000,  8026,  1000,  2001,
         19499, 19142,  1010,  2021,  1010, 16267, 20857,  1012,  1026,  7987,
          1013,  1028,  1026,  7987,  1013,  1028,  1996,  4955,  1997,  1996,
          2839,  1010,  2209,  2011,  3533,  6090,  3406, 15204,  2080,  1011,
          1996,  2502,  3066,  3185,  3124,  1010,  2008,  3268,  1999,  1996,
          2380,  1998, 25126,  1999,  1037, 13697,  7062,  1010,  3253,  3246,
          1998,  1045,  2787,  2000,  2507,  2009,  1037,  2261,  2062,  2781,
          1012,  1998,  2059,  1037,  2261,  2062,  2127, 19031,  3723, 26699,
          5644,  

# Fine-tune DistilBERT

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained( # PyTorch-модель
    "distilbert-base-uncased",
    num_labels=2 # сообщаем модели сколько классов
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred # logits — “сырые” выходы модели (до softmax)
    preds = np.argmax(logits, axis=-1) # выбираем класс с максимальным логитом \ axis — это “по какой оси массива мы делаем операцию”. axis=1 → работаем “по строкам” (вправо)
    return accuracy_metric.compute(predictions=preds, references=labels) # предсказания и истинные метки

In [ ]:
# параметры обучения

training_args = TrainingArguments(
    output_dir="./distilbert-imdb",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

*   output_dir: куда сохранять чекпоинты
*   eval_strategy="epoch": после каждой эпохи считать метрики на test
*   save_strategy="epoch": сохранять модель после каждой эпохи
*   learning_rate=2e-5: шаг обучения (типичное значение для BERT fine-tuning)
*   batch_size: сколько примеров за один шаг градиента
*   num_train_epochs=2: сколько раз пройти весь train
*   weight_decay: “штраф” за большие веса (уменьшает переобучение)
*   load_best_model_at_end: загрузить лучший чекпоинт по метрике (в конце забрать лучшую модель по метрике)
*   metric_for_best_model="accuracy" — по какой метрике считать лучшей







In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"], # данные, на которых модель учится (обновляет веса)
    eval_dataset=tokenized_dataset["test"], # данные, на которых модель проверяется (метрики), но веса не обновляются
    compute_metrics=compute_metrics # оценка метрик на каждой эпохе
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.296055,0.250184,0.903500
2,0.196018,0.256030,0.910000


[[-0.63349605  0.5496173 ]
 [ 1.5996417  -1.4415591 ]
 [-1.890601    1.578233  ]
 [ 0.16963686 -0.24794303]
 [ 0.4263155  -0.2639038 ]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[[-1.370367   1.2154441]
 [ 2.2061422 -1.9626343]
 [-2.4048114  2.0812526]
 [-1.2082018  0.9554195]
 [ 0.6763064 -0.3771808]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=626, training_loss=0.29193461817293503, metrics={'train_runtime': 245.8804, 'train_samples_per_second': 40.67, 'train_steps_per_second': 2.546, 'total_flos': 662336993280000.0, 'train_loss': 0.29193461817293503, 'epoch': 2.0})

In [ ]:
print("CUDA доступен:", torch.cuda.is_available())
print("Устройство модели:", trainer.model.device)

CUDA доступен: True
Устройство модели: cuda:0


In [ ]:
results = trainer.evaluate()
results

[[-1.370367   1.2154441]
 [ 2.2061422 -1.9626343]
 [-2.4048114  2.0812526]
 [-1.2082018  0.9554195]
 [ 0.6763064 -0.3771808]]


{'eval_loss': 0.2560296654701233,
 'eval_accuracy': 0.91,
 'eval_runtime': 13.0603,
 'eval_samples_per_second': 153.135,
 'eval_steps_per_second': 4.824,
 'epoch': 2.0}

In [ ]:
#'eval_loss': мера ошибки модели,
#'eval_accuracy': точность модели,
#'eval_runtime': Сколько секунд заняла оценка на test,
#'eval_samples_per_second': колько примеров модель обрабатывает в секунду,
#'eval_steps_per_second': batch-а в секунду,
#'epoch': сколько полных проходов по датасету

In [ ]:
model.eval()

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
import torch.nn.functional as F

def predict(text):
    inputs = tokenizer( # inputs["input_ids"] будет типа torch.Tensor
        text,
        return_tensors="pt", # вернуть тензоры PyTorch
        truncation=True, # обрезка
        padding="max_length",
        max_length=256
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()} # каждый тензор из inputs переносится на model.device (туда же где лежит модель) \ inputs словарь

    with torch.no_grad(): # отключает вычисление градиентов
        outputs = model(**inputs)

    logits = outputs.logits
    probs = F.softmax(logits, dim=-1)
    pred_class = torch.argmax(probs, dim=-1).item() # .item() преобразует тензор из одного числа в обычный Python int. (если у тензора единичное значение)
    confidence = probs[0][pred_class].item()

    return pred_class, confidence

In [ ]:
print("=== ПОЛОЖИТЕЛЬНЫЕ ОТЗЫВЫ ===\n")

count = 0
for example in small_dataset["test"]:
    if example["label"] == 1:
        pred, conf = predict(example["text"])

        print("Text:", example["text"][:300], "...\n")
        print("True label:", example["label"])
        print("Predicted:", pred)
        print("Confidence:", round(conf, 4))
        print("-"*70)

        count += 1
        if count == 5:
            break

=== ПОЛОЖИТЕЛЬНЫЕ ОТЗЫВЫ ===

Text: Despite its pedigree, the most interesting things about this series are not the animatronics or puppetry, which, while charming, are little more than sideshows, at least in the story I saw, A STORY SHORT. In fact, loathe though I am to admit it, the programme's chief pleasure lies in that most ancie ...

True label: 1
Predicted: 1
Confidence: 0.9299
----------------------------------------------------------------------
Text: Being a "Wallace and Gromit-fan", I was looking forward for this full-length movie. Surprisingly I saw it at THE world-premiere in Vlissingen (NL), at the Film by the Sea festival. A wonderful feeling to be one of the first to see this very amusing and merry movie. It's about Wallace and Gromit (who ...

True label: 1
Predicted: 1
Confidence: 0.9889
----------------------------------------------------------------------
Text: "Best in Show" is a often hilarious mockumentary that takes us into the world of dog shows and some of the

In [ ]:
print("\n=== НЕГАТИВНЫЕ ОТЗЫВЫ ===\n")

count = 0
for example in small_dataset["test"]:
    if example["label"] == 0:
        pred, conf = predict(example["text"])

        print("TEXT:", example["text"][:300], "...\n")
        print("True label:", example["label"])
        print("Predicted:", pred)
        print("Confidence:", round(conf, 4))
        print("-"*70)

        count += 1
        if count == 5:
            break


=== НЕГАТИВНЫЕ ОТЗЫВЫ ===

TEXT: This is a movie with an excellent concept for a story but that got sidetracked but a large number of clichéd sub-plots, hackneyed and unrealistic portrayed characterizations and performances, and some frankly implausible (and highly coincidental and, not to mention, convenient as plot points to move ...

True label: 0
Predicted: 0
Confidence: 0.9848
----------------------------------------------------------------------
TEXT: I watched this movie for the first time the other day and was bored to tears. I guess I just was looking for some flashback to the wonderful series that I remembered. I watched The Mod Squad television show religiously back in the day and it was fantastic. It was action packed and the relationship t ...

True label: 0
Predicted: 1
Confidence: 0.8969
----------------------------------------------------------------------
TEXT: So, this starts with at least an interesting and promising basic idea, goes on and on with tension, Carey in